# Problemas Potenciales en Regresión Lineal

Notebook del módulo 10. Cubre los seis problemas clásicos de OLS:
no-linealidad, correlación de errores, heterocedasticidad, outliers,
alto leverage y colinealidad.

Relacionado con módulo 06 (regresion_lib.ipynb) donde se introdujeron los supuestos de OLS.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import OLSInfluence, variance_inflation_factor
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#0f0f11',
    'axes.facecolor':   '#0f0f11',
    'axes.edgecolor':   '#2a2a30',
    'axes.labelcolor':  '#8a8a9a',
    'xtick.color':      '#8a8a9a',
    'ytick.color':      '#8a8a9a',
    'text.color':       '#e2e2e8',
    'grid.color':       '#2a2a30',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
})
ACCENT = '#6c7fea'
RED    = '#ea6c6c'
TEXT2  = '#8a8a9a'

df = pd.read_csv('../data/csvs/examenes.csv').sample(n=200, random_state=42)
study_hours = np.array(df['study_hours'])
exam_score  = np.array(df['exam_score'])
print(df.shape)
df.head()

## 1. No-linealidad

Si la relación entre predictores y respuesta no es lineal, los residuos
del modelo mostrarán un patrón en arco en la gráfica de residuos vs ajustados.

In [ ]:
# Ajuste lineal vs cuadrático
X_lin  = sm.add_constant(study_hours)
m_lin  = sm.OLS(exam_score, X_lin).fit()

X_poly = sm.add_constant(np.column_stack([study_hours, study_hours**2]))
m_poly = sm.OLS(exam_score, X_poly).fit()

print(f'R² lineal:      {m_lin.rsquared:.3f}')
print(f'R² cuadrático: {m_poly.rsquared:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Residuos vs valores ajustados', fontsize=12)

for ax, model, title in [
    (axes[0], m_lin,  'Modelo lineal'),
    (axes[1], m_poly, 'Modelo cuadrático'),
]:
    ax.scatter(model.fittedvalues, model.resid, color=ACCENT, alpha=0.5, s=18, edgecolors='none')
    ax.axhline(0, color=RED, linestyle='--', linewidth=1.5)
    ax.set_title(title)
    ax.set_xlabel('Valores ajustados ŷ')
    ax.set_ylabel('Residuos')
    ax.grid(True)

plt.tight_layout()
plt.show()

## 2. Correlación de errores

Usamos Durbin-Watson para detectar autocorrelación en los residuos.
DW ≈ 2 → sin correlación. DW < 1.5 o > 2.5 → alerta.

In [ ]:
# Ordenar por class_attendance como proxy de orden temporal
df_sorted = df.sort_values('class_attendance').reset_index(drop=True)
X_s = sm.add_constant(df_sorted['study_hours'])
m_s = sm.OLS(df_sorted['exam_score'], X_s).fit()

dw = durbin_watson(m_s.resid)
print(f'Durbin-Watson: {dw:.3f}')
print('Interpretación:', 'sin autocorrelación' if 1.5 < dw < 2.5 else 'posible autocorrelación')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(m_s.resid.values, color=ACCENT, linewidth=0.8, alpha=0.8)
ax.axhline(0, color=RED, linestyle='--', linewidth=1.5)
ax.set_title(f'Residuos en orden de observación  (DW = {dw:.2f})')
ax.set_xlabel('Índice de observación')
ax.set_ylabel('Residuo')
ax.grid(True)
plt.tight_layout()
plt.show()

## 3. Heterocedasticidad

La varianza de los residuos no es constante — aparece como un embudo
en la gráfica de residuos vs ajustados. La transformación log suele corregirlo.

In [ ]:
# Datos sintéticos con heterocedasticidad pronunciada
rng = np.random.RandomState(42)
x_h = np.linspace(1, 10, 200)
y_h = 3 * x_h + rng.normal(0, 1, 200) * x_h   # σ crece con x

m_orig = sm.OLS(y_h, sm.add_constant(x_h)).fit()

y_log  = np.log(y_h - y_h.min() + 1)
m_log  = sm.OLS(y_log, sm.add_constant(x_h)).fit()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Heterocedasticidad', fontsize=12)

for ax, model, title in [
    (axes[0], m_orig, 'Escala original — embudo'),
    (axes[1], m_log,  'Tras transformación log'),
]:
    ax.scatter(model.fittedvalues, model.resid, color=ACCENT, alpha=0.5, s=18, edgecolors='none')
    ax.axhline(0, color=RED, linestyle='--', linewidth=1.5)
    ax.set_title(title)
    ax.set_xlabel('Valores ajustados ŷ')
    ax.set_ylabel('Residuos')
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Transformación log en datos reales
y_log_real = np.log1p(exam_score)
m_log_real = sm.OLS(y_log_real, sm.add_constant(study_hours)).fit()

b1 = m_log_real.params[1]
print(f'R² con log(y): {m_log_real.rsquared:.3f}')
print(f'Una hora más de estudio → {b1*100:.1f}% de cambio en calificación')

## 4. Outliers

Una observación con respuesta inusual puede desplazar la recta.
Los residuos estudentizados con |r_i| > 3 son candidatos a outlier.

In [ ]:
# Inyectar dos outliers artificiales
y_mod = exam_score.copy()
y_mod[0]  = exam_score.mean() + 4 * exam_score.std()
y_mod[10] = exam_score.mean() - 4 * exam_score.std()

m_clean = sm.OLS(exam_score, sm.add_constant(study_hours)).fit()
m_out   = sm.OLS(y_mod,      sm.add_constant(study_hours)).fit()

print(f'Pendiente sin outliers: {m_clean.params[1]:.3f}')
print(f'Pendiente con outliers: {m_out.params[1]:.3f}')

In [ ]:
# Residuos estudentizados
r_stud = OLSInfluence(m_out).resid_studentized_external

print(f'Observaciones con |r_i| > 3: {(np.abs(r_stud) > 3).sum()}')
print(f'Observaciones con |r_i| > 2: {(np.abs(r_stud) > 2).sum()}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(range(len(r_stud)), r_stud, color=ACCENT, alpha=0.6, s=20, edgecolors='none')
ax.axhline( 3, color=RED, linestyle='--', linewidth=1.5, label='|r| = 3')
ax.axhline(-3, color=RED, linestyle='--', linewidth=1.5)
outlier_idx = np.where(np.abs(r_stud) > 3)[0]
ax.scatter(outlier_idx, r_stud[outlier_idx], color=RED, s=60, zorder=5)
ax.set_title('Residuos estudentizados')
ax.set_xlabel('Índice de observación')
ax.set_ylabel('Residuo estudentizado')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## 5. Alto leverage

Un punto de alto leverage tiene un valor de x muy alejado de la media.
Umbral práctico: h_ii > 2(p+1)/n

In [ ]:
modelo_base = sm.OLS(exam_score, sm.add_constant(study_hours)).fit()
infl  = OLSInfluence(modelo_base)
h_vals = infl.hat_matrix_diag

p = 1
n = len(exam_score)
umbral = 2 * (p + 1) / n

print(f'Umbral de leverage 2(p+1)/n = {umbral:.3f}')
print(f'Puntos con h_ii > umbral: {(h_vals > umbral).sum()}')

In [ ]:
# Cook's distance
cook_d, _ = infl.cooks_distance
print(f'Observaciones con Cook D > 1: {(cook_d > 1).sum()}')
print(f'Cook D máximo: {cook_d.max():.4f}')

## 6. Colinealidad

Cuando dos o más predictores están correlacionados, los errores estándar
se inflan y los p-valores dejan de ser confiables.

In [ ]:
# Modelo con tres predictores
vars_pred = ['study_hours', 'class_attendance', 'sleep_hours']
X_multi = sm.add_constant(df[vars_pred])
m_multi = sm.OLS(df['exam_score'], X_multi).fit()
print(m_multi.summary())

In [ ]:
# VIF de cada predictor
X_vif = sm.add_constant(df[vars_pred].values)

print('Factor de Inflación de Varianza (VIF):')
for i, var in enumerate(vars_pred):
    vif = variance_inflation_factor(X_vif, i + 1)
    estado = 'OK' if vif < 5 else ('Moderado' if vif < 10 else 'SEVERO')
    print(f'  {var:22s}: {vif:.2f}  [{estado}]')

In [ ]:
# Comparar errores estándar con y sin colinealidad
X_col = sm.add_constant(df[['study_hours', 'class_attendance']])
m_col = sm.OLS(df['exam_score'], X_col).fit()

X_sin = sm.add_constant(df[['study_hours']])
m_sin = sm.OLS(df['exam_score'], X_sin).fit()

print('Errores estándar de study_hours:')
print(f'  Con class_attendance (colineal): {m_col.bse["study_hours"]:.4f}')
print(f'  Sin class_attendance:            {m_sin.bse["study_hours"]:.4f}')

## Resumen diagnóstico

| Problema | Herramienta | Umbral de alerta |
|---|---|---|
| No-linealidad | Gráfica residuos vs ŷ | Patrón visible |
| Correlación de errores | Durbin-Watson | DW < 1.5 o DW > 2.5 |
| Heterocedasticidad | Gráfica residuos vs ŷ | Forma de embudo |
| Outliers | Residuos estudentizados | |r_i| > 3 |
| Alto leverage | hat_matrix_diag | h_ii > 2(p+1)/n |
| Colinealidad | VIF | VIF > 10 |